In [7]:
import pandas as pd
import zipfile
import urllib.request
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches

In [8]:

# Download and extract MovieLens 100k dataset
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
dataset_zip = "ml-100k.zip"
dataset_folder = "ml-100k"
data_file = os.path.join(dataset_folder, "u.item")

if not os.path.exists(data_file):
    print("Downloading dataset...")
    urllib.request.urlretrieve(dataset_url, dataset_zip)

    print("Extracting dataset...")
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall()

    print("Dataset ready!")



In [9]:
# Load movie metadata
columns = [
    'movie_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
    'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movies = pd.read_csv(data_file, sep='|', names=columns, encoding='latin-1')

# Combine genres into single string
genre_columns = columns[5:]
movies['genres'] = movies[genre_columns].apply(
    lambda row: ' '.join([genre for genre, val in zip(genre_columns, row) if val == 1]),
    axis=1
)



In [10]:
movies

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,genres
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,Animation Children's Comedy
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,1,0,0,Action Adventure Thriller
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,Thriller
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,Action Comedy Drama
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,Crime Drama Thriller
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Drama
1678,1679,B. Monkey (1998),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,Romance Thriller
1679,1680,Sliding Doors (1998),01-Jan-1998,NaN,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Drama Romance
1680,1681,You So Crazy (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Comedy


In [12]:
# Vectorize genres
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(movies['genres'])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Enhanced recommendation with fuzzy match
def recommend_movies(title_input, df, sim_matrix, top_n=5):
    titles = df['title'].tolist()
    matches = get_close_matches(title_input, titles, n=1, cutoff=0.5)

    if not matches:
        return "Movie not found. Try typing more of the title or check spelling."

    matched_title = matches[0]
    idx = df[df['title'] == matched_title].index[0]

    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return matched_title, df.iloc[top_indices][['title', 'genres']]



In [15]:
print("\n Smart Movie Recommender (Enter partial title too!)")
user_input = input("Enter a movie title (e.g. star wars): ")

result = recommend_movies(user_input, movies, cosine_sim)

if isinstance(result, str):
    print(result)
else:
    matched, recommendations = result
    print(f"\n Closest match: {matched}")
    print("\n Recommendations:\n")
    print(recommendations.to_string(index=False))


 Smart Movie Recommender (Enter partial title too!)
Enter a movie title (e.g. star wars): Toy story

 Closest match: Toy Story (1995)

 Recommendations:

                                 title                      genres
Aladdin and the King of Thieves (1996) Animation Children's Comedy
                Aristocats, The (1970)        Animation Children's
                      Pinocchio (1940)        Animation Children's
        Sword in the Stone, The (1963)        Animation Children's
         Fox and the Hound, The (1981)        Animation Children's
